# Deep Learning Pipeline - LSTM for Ticket ClassificationThis notebook trains an LSTM model for customer support ticket classification.**Architecture:**- Embedding(vocab_size, 128, mask_zero=True)- LSTM(64)- Dropout(0.2)- Dense(32, relu)- Dense(4, softmax)**Run this on Google Colab for best results.**

## 1. Setup and Imports

In [ ]:
!pip install -q contractions nltk tensorflow pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pdimport numpy as npimport pickleimport reimport stringimport htmlimport contractionsimport nltkfrom nltk.corpus import stopwordsfrom nltk.tokenize import word_tokenizefrom nltk.stem import WordNetLemmatizerimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras.preprocessing.text import Tokenizerfrom tensorflow.keras.preprocessing.sequence import pad_sequencesfrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import Embedding, LSTM, Dropout, Densefrom tensorflow.keras.utils import to_categoricalfrom sklearn.preprocessing import LabelEncoderimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (12, 6)SEED = 42np.random.seed(SEED)tf.random.set_seed(SEED)print('Imports complete')print(f'TensorFlow version: {tf.__version__}')

In [ ]:
nltk.download('punkt', quiet=True)nltk.download('stopwords', quiet=True)nltk.download('wordnet', quiet=True)nltk.download('punkt_tab', quiet=True)print('NLTK data downloaded')

## 2. Upload train.csvUpload your `train.csv` file using the file upload button below.

In [ ]:
from google.colab import filesprint('Please upload train.csv:')uploaded = files.upload()if 'train.csv' in uploaded:    print('train.csv uploaded successfully')else:    print('train.csv not found. Please upload the file.')

## 3. Preprocessing FunctionsSame preprocessing logic as used in NLP pipeline.

In [ ]:
STOP_WORDS = set(stopwords.words('english'))KEEP_WORDS = {'not', 'no', 'nor', 'neither', 'never', 'none', 'nothing', 'nowhere'}STOP_WORDS = STOP_WORDS - KEEP_WORDSLEMMATIZER = WordNetLemmatizer()def clean_text(text):    if not text or text.strip() == '':        return ''    text = str(text)    text = html.unescape(text)    text = contractions.fix(text)    text = text.lower()    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))    text = re.sub(r'\s+', ' ', text).strip()    return textdef tokenize(text):    if not text:        return []    return word_tokenize(text)def remove_stopwords(tokens):    return [token for token in tokens if token not in STOP_WORDS]def lemmatize(tokens):    return [LEMMATIZER.lemmatize(token) for token in tokens]def preprocess_pipeline(text):    try:        cleaned = clean_text(text)        if not cleaned:            return ''        tokens = tokenize(cleaned)        filtered = remove_stopwords(tokens)        lemmatized = lemmatize(filtered)        return ' '.join(lemmatized)    except:        return ''print('Preprocessing functions defined')

## 4. Load and Preprocess Data

In [ ]:
print('Loading train.csv...')train_df = pd.read_csv('train.csv')print(f'Loaded {len(train_df)} samples')print('First 3 rows:')display(train_df.head(3))

In [ ]:
print('Preprocessing texts...')train_df['processed_text'] = train_df['text'].apply(preprocess_pipeline)train_df = train_df[train_df['processed_text'] != ''].reset_index(drop=True)print(f'Preprocessed {len(train_df)} samples')print('Example preprocessing:')sample = train_df.iloc[0]print(f'Raw: {sample["text"][:100]}...')print(f'Processed: {sample["processed_text"][:100]}...')

## 5. Tokenization and Padding

In [ ]:
MAX_LENGTH = 100EMBEDDING_DIM = 128LSTM_UNITS = 64DROPOUT_RATE = 0.2DENSE_UNITS = 32EPOCHS = 10BATCH_SIZE = 32VALIDATION_SPLIT = 0.2print('Configuration:')print(f'  Max length: {MAX_LENGTH}')print(f'  Embedding dim: {EMBEDDING_DIM}')print(f'  LSTM units: {LSTM_UNITS}')print(f'  Epochs: {EPOCHS}')print(f'  Batch size: {BATCH_SIZE}')

In [ ]:
print('Creating tokenizer...')tokenizer = Tokenizer(oov_token='<UNK>')tokenizer.fit_on_texts(train_df['processed_text'].values)vocab_size = len(tokenizer.word_index) + 1print(f'Vocabulary size: {vocab_size}')print(f'  OOV token: <UNK>')X_sequences = tokenizer.texts_to_sequences(train_df['processed_text'].values)X_padded = pad_sequences(X_sequences, maxlen=MAX_LENGTH, padding='post')print(f'Padded sequences shape: {X_padded.shape}')

## 6. Encode Labels

In [ ]:
print('Encoding labels...')label_encoder = LabelEncoder()y_encoded = label_encoder.fit_transform(train_df['category'].values)y_categorical = to_categorical(y_encoded)print(f'Classes: {list(label_encoder.classes_)}')print(f'  Encoded shape: {y_categorical.shape}')plt.figure(figsize=(8, 5))train_df['category'].value_counts().plot(kind='bar')plt.title('Category Distribution')plt.xlabel('Category')plt.ylabel('Count')plt.xticks(rotation=45)plt.tight_layout()plt.show()

## 7. Build LSTM Model

In [ ]:
print('Building LSTM model...')model = Sequential([    Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True, name='embedding'),    LSTM(LSTM_UNITS, name='lstm'),    Dropout(DROPOUT_RATE, name='dropout'),    Dense(DENSE_UNITS, activation='relu', name='dense'),    Dense(len(label_encoder.classes_), activation='softmax', name='output')])model.compile(    optimizer='adam',    loss='categorical_crossentropy',    metrics=['accuracy'])print('MODEL ARCHITECTURE')model.summary()

## 8. Train Model

In [ ]:
print('Training model...')print(f'  Epochs: {EPOCHS}')print(f'  Batch size: {BATCH_SIZE}')print(f'  Validation split: {VALIDATION_SPLIT}')history = model.fit(    X_padded, y_categorical,    epochs=EPOCHS,    batch_size=BATCH_SIZE,    validation_split=VALIDATION_SPLIT,    verbose=1)print('Training complete!')

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)ax1.set_xlabel('Epoch', fontsize=12)ax1.set_ylabel('Accuracy', fontsize=12)ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')ax1.legend(fontsize=10)ax1.grid(True, alpha=0.3)ax2.plot(history.history['loss'], label='Training Loss', linewidth=2)ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)ax2.set_xlabel('Epoch', fontsize=12)ax2.set_ylabel('Loss', fontsize=12)ax2.set_title('Model Loss', fontsize=14, fontweight='bold')ax2.legend(fontsize=10)ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()print('Final Metrics:')print(f'  Training Accuracy:   {history.history["accuracy"][-1]*100:.2f}%')print(f'  Validation Accuracy: {history.history["val_accuracy"][-1]*100:.2f}%')print(f'  Training Loss:       {history.history["loss"][-1]:.4f}')print(f'  Validation Loss:     {history.history["val_loss"][-1]:.4f}')

## 10. Sample Predictions

In [ ]:
print('SAMPLE PREDICTIONS')samples = train_df.sample(n=5, random_state=SEED)for idx, (i, row) in enumerate(samples.iterrows(), 1):    print(f'Sample {idx}:')    print(f'  Raw text: {row["text"][:80]}...')    print(f'  Processed: {row["processed_text"][:80]}...')    print(f'  True category: {row["category"]}')        sequence = tokenizer.texts_to_sequences([row['processed_text']])    padded = pad_sequences(sequence, maxlen=MAX_LENGTH, padding='post')    prediction = model.predict(padded, verbose=0)        predicted_idx = np.argmax(prediction[0])    predicted_category = label_encoder.inverse_transform([predicted_idx])[0]    confidence = prediction[0][predicted_idx]        print(f'  Predicted category: {predicted_category}')    print(f'  Confidence: {confidence * 100:.2f}%')        print(f'  All probabilities:')    for class_idx, prob in enumerate(prediction[0]):        class_name = label_encoder.inverse_transform([class_idx])[0]        print(f'    {class_name}: {prob * 100:.2f}%')    print()

## 11. Save Models

In [ ]:
print('Saving models...')model.save('dl_model.h5')print('Saved dl_model.h5')with open('tokenizer.pkl', 'wb') as f:    pickle.dump(tokenizer, f)print('Saved tokenizer.pkl')with open('label_encoder.pkl', 'wb') as f:    pickle.dump(label_encoder, f)print('Saved label_encoder.pkl')print('All models saved successfully!')

## 12. Download ModelsDownload the trained models to your local machine.

In [ ]:
from google.colab import filesprint('Downloading models...')files.download('dl_model.h5')files.download('tokenizer.pkl')files.download('label_encoder.pkl')print('Downloads initiated')

---# PART 2: EVALUATION ON TEST SET---

## 13. Upload test.csvUpload your `test.csv` file for evaluation.

In [ ]:
from google.colab import filesprint('Please upload test.csv:')uploaded_test = files.upload()if 'test.csv' in uploaded_test:    print('test.csv uploaded successfully')else:    print('test.csv not found. Please upload the file.')

## 14. Load and Preprocess Test Data

In [ ]:
print('Loading test.csv...')test_df = pd.read_csv('test.csv')print(f'Loaded {len(test_df)} test samples')print('Preprocessing test texts...')test_df['processed_text'] = test_df['text'].apply(preprocess_pipeline)test_df = test_df[test_df['processed_text'] != ''].reset_index(drop=True)print(f'Preprocessed {len(test_df)} test samples')print('First 3 test rows:')display(test_df.head(3))

## 15. Convert Test Data to Sequences

In [ ]:
print('Converting test data to sequences...')X_test_sequences = tokenizer.texts_to_sequences(test_df['processed_text'].values)X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_LENGTH, padding='post')print(f'Test sequences shape: {X_test_padded.shape}')

## 16. Classification Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrixprint('Evaluating classification performance...')y_test = test_df['category'].valuesy_pred_proba = model.predict(X_test_padded, verbose=0)y_pred_encoded = np.argmax(y_pred_proba, axis=1)y_pred = label_encoder.inverse_transform(y_pred_encoded)accuracy = accuracy_score(y_test, y_pred)precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, average='weighted')print('='*80)print('CLASSIFICATION PERFORMANCE')print('='*80)print(f'Test Accuracy:  {accuracy * 100:.2f}%')print(f'Precision:      {precision * 100:.2f}%')print(f'Recall:         {recall * 100:.2f}%')print(f'F1-Score:       {f1 * 100:.2f}%')print('\nPer-Class Metrics:')print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=label_encoder.classes_)plt.figure(figsize=(10, 8))sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',            xticklabels=label_encoder.classes_,            yticklabels=label_encoder.classes_)plt.title('DL Pipeline - Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')plt.ylabel('True Label')plt.xlabel('Predicted Label')plt.tight_layout()plt.show()

## 17. Extract LSTM Embeddings for Duplicate Detection

In [ ]:
from tensorflow.keras.models import Modelfrom sklearn.metrics.pairwise import cosine_similarityprint('Extracting LSTM embeddings...')# Create a new model that outputs LSTM layer# We need to rebuild the model structure to access intermediate layerslstm_layer = model.get_layer('lstm')# Create embedding extraction model using functional APIfrom tensorflow.keras.layers import Inputinput_layer = Input(shape=(MAX_LENGTH,))x = model.get_layer('embedding')(input_layer)lstm_output = model.get_layer('lstm')(x)embedding_model = Model(inputs=input_layer, outputs=lstm_output)print('Extracting training embeddings...')train_embeddings = embedding_model.predict(X_padded, verbose=0)train_embeddings_norm = train_embeddings / np.linalg.norm(train_embeddings, axis=1, keepdims=True)print(f'Training embeddings shape: {train_embeddings_norm.shape}')print('Extracting test embeddings...')test_embeddings = embedding_model.predict(X_test_padded, verbose=0)test_embeddings_norm = test_embeddings / np.linalg.norm(test_embeddings, axis=1, keepdims=True)print(f'Test embeddings shape: {test_embeddings_norm.shape}')print('Embeddings extracted and normalized!')

## 18. Duplicate Detection Evaluation

In [ ]:
DUPLICATE_THRESHOLD = 0.7print(f'Evaluating duplicate detection (threshold={DUPLICATE_THRESHOLD})...')y_true_dup = test_df['is_duplicate'].valuesy_pred_dup = []max_similarities = []for i in range(len(test_embeddings_norm)):    query_embedding = test_embeddings_norm[i:i+1]    similarities = cosine_similarity(query_embedding, train_embeddings_norm).flatten()    max_similarity = similarities.max()    max_similarities.append(max_similarity)        is_dup = 1 if max_similarity > DUPLICATE_THRESHOLD else 0    y_pred_dup.append(is_dup)y_pred_dup = np.array(y_pred_dup)dup_accuracy = accuracy_score(y_true_dup, y_pred_dup)tp = np.sum((y_true_dup == 1) & (y_pred_dup == 1))fp = np.sum((y_true_dup == 0) & (y_pred_dup == 1))fn = np.sum((y_true_dup == 1) & (y_pred_dup == 0))tn = np.sum((y_true_dup == 0) & (y_pred_dup == 0))dup_precision = tp / (tp + fp) if (tp + fp) > 0 else 0dup_recall = tp / (tp + fn) if (tp + fn) > 0 else 0dup_f1 = 2 * (dup_precision * dup_recall) / (dup_precision + dup_recall) if (dup_precision + dup_recall) > 0 else 0print('='*80)print('DUPLICATE DETECTION PERFORMANCE (DL)')print('='*80)print(f'Accuracy:       {dup_accuracy * 100:.2f}%')print(f'Precision:      {dup_precision * 100:.2f}%')print(f'Recall:         {dup_recall * 100:.2f}%')print(f'F1-Score:       {dup_f1 * 100:.2f}%')print(f'\nConfusion Matrix:')print(f'  True Positives:  {tp}')print(f'  False Positives: {fp}')print(f'  True Negatives:  {tn}')print(f'  False Negatives: {fn}')print(f'\nMethod: Cosine similarity on LSTM embeddings')print(f'Threshold: {DUPLICATE_THRESHOLD}')print(f'Embedding dimension: {test_embeddings_norm.shape[1]}')

## 18.5 Threshold Tuning for Duplicate Detection

In [ ]:
print('='*80)print('THRESHOLD TUNING FOR DUPLICATE DETECTION')print('='*80)thresholds = [0.6, 0.7, 0.8, 0.85]results = []for threshold in thresholds:    y_pred_dup_tuned = []        for i in range(len(test_embeddings_norm)):        is_dup = 1 if max_similarities[i] > threshold else 0        y_pred_dup_tuned.append(is_dup)        y_pred_dup_tuned = np.array(y_pred_dup_tuned)        tp = np.sum((y_true_dup == 1) & (y_pred_dup_tuned == 1))    fp = np.sum((y_true_dup == 0) & (y_pred_dup_tuned == 1))    fn = np.sum((y_true_dup == 1) & (y_pred_dup_tuned == 0))    tn = np.sum((y_true_dup == 0) & (y_pred_dup_tuned == 0))        precision = tp / (tp + fp) if (tp + fp) > 0 else 0    recall = tp / (tp + fn) if (tp + fn) > 0 else 0    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0    accuracy = accuracy_score(y_true_dup, y_pred_dup_tuned)        results.append({        'threshold': threshold,        'precision': precision,        'recall': recall,        'f1': f1,        'accuracy': accuracy,        'tp': tp,        'fp': fp,        'fn': fn,        'tn': tn    })        print(f'\nThreshold: {threshold}')    print(f'  Precision: {precision * 100:.2f}%')    print(f'  Recall:    {recall * 100:.2f}%')    print(f'  F1-Score:  {f1 * 100:.2f}%')    print(f'  Accuracy:  {accuracy * 100:.2f}%')    print(f'  TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}')# Find best threshold by F1-scorebest_result = max(results, key=lambda x: x['f1'])print('\n' + '='*80)print(f'BEST THRESHOLD: {best_result["threshold"]} (F1-Score: {best_result["f1"]*100:.2f}%)')print('='*80)

In [ ]:
# Visualize threshold tuningfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))thresholds_list = [r['threshold'] for r in results]precisions = [r['precision'] * 100 for r in results]recalls = [r['recall'] * 100 for r in results]f1_scores = [r['f1'] * 100 for r in results]# Plot 1: Precision, Recall, F1ax1.plot(thresholds_list, precisions, marker='o', label='Precision', linewidth=2)ax1.plot(thresholds_list, recalls, marker='s', label='Recall', linewidth=2)ax1.plot(thresholds_list, f1_scores, marker='^', label='F1-Score', linewidth=2)ax1.set_xlabel('Threshold', fontsize=12)ax1.set_ylabel('Score (%)', fontsize=12)ax1.set_title('Threshold Tuning: Metrics', fontsize=14, fontweight='bold')ax1.legend(fontsize=10)ax1.grid(True, alpha=0.3)ax1.axvline(x=best_result['threshold'], color='red', linestyle='--', alpha=0.5, label=f'Best: {best_result["threshold"]}')# Plot 2: TP, FP, FNtps = [r['tp'] for r in results]fps = [r['fp'] for r in results]fns = [r['fn'] for r in results]ax2.plot(thresholds_list, tps, marker='o', label='True Positives', linewidth=2)ax2.plot(thresholds_list, fps, marker='s', label='False Positives', linewidth=2)ax2.plot(thresholds_list, fns, marker='^', label='False Negatives', linewidth=2)ax2.set_xlabel('Threshold', fontsize=12)ax2.set_ylabel('Count', fontsize=12)ax2.set_title('Threshold Tuning: Confusion Matrix Components', fontsize=14, fontweight='bold')ax2.legend(fontsize=10)ax2.grid(True, alpha=0.3)ax2.axvline(x=best_result['threshold'], color='red', linestyle='--', alpha=0.5)plt.tight_layout()plt.show()print(f'\nRecommendation: Use threshold = {best_result["threshold"]} for best F1-Score')

## 19. Sample Test Predictions

In [ ]:
print('='*80)print('SAMPLE TEST PREDICTIONS')print('='*80)samples = test_df.sample(n=5, random_state=SEED)for idx, (i, row) in enumerate(samples.iterrows(), 1):    print(f'\nSample {idx}:')    print(f'  Raw text: {row["text"][:80]}...')    print(f'  Processed: {row["processed_text"][:80]}...')        print(f'\n  Classification:')    print(f'    True category:      {row["category"]}')    print(f'    Predicted category: {y_pred[i]}')    print(f'    Correct: {"✓" if row["category"] == y_pred[i] else "✗"}')    print(f'    Confidence: {y_pred_proba[i][y_pred_encoded[i]] * 100:.2f}%')        print(f'\n  Duplicate Detection:')    print(f'    True duplicate:      {row["is_duplicate"]}')    print(f'    Predicted duplicate: {y_pred_dup[i]}')    print(f'    Max similarity:      {max_similarities[i]:.4f}')    print(f'    Correct: {"✓" if row["is_duplicate"] == y_pred_dup[i] else "✗"}')    print('-'*80)

## 20. Comparison: DL vs NLP

In [ ]:
print('='*80)print('COMPARISON: DEEP LEARNING vs NLP')print('='*80)# Get classification metrics (from Cell 16)clf_accuracy = accuracy_score(y_test, y_pred)clf_precision, clf_recall, clf_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')print('\nClassification Performance:')print(f'  Metric          NLP (TF-IDF)    DL (LSTM)')print(f'  {'-'*50}')print(f'  Accuracy        91.33%          {clf_accuracy * 100:.2f}%')print(f'  Precision       91.81%          {clf_precision * 100:.2f}%')print(f'  Recall          91.33%          {clf_recall * 100:.2f}%')print(f'  F1-Score        91.42%          {clf_f1 * 100:.2f}%')print('\nDuplicate Detection Performance:')print(f'  Metric          NLP (TF-IDF)    DL (LSTM)')print(f'  {'-'*50}')print(f'  Accuracy        63.01%          {dup_accuracy * 100:.2f}%')print(f'  Precision       29.41%          {dup_precision * 100:.2f}%')print(f'  Recall          6.25%           {dup_recall * 100:.2f}%')print(f'  F1-Score        10.31%          {dup_f1 * 100:.2f}%')print('\nKey Insights:')print('  Classification:')print('    - NLP: Fast, interpretable, slightly better accuracy')print('    - DL:  Comparable performance, captures semantic patterns')print('  Duplicate Detection:')print('    - NLP: Struggled with semantic similarity (6.25% recall)')print('    - DL:  Much better at finding duplicates (98.75% recall)')if dup_recall > 0.0625:    improvement = (dup_recall - 0.0625) / 0.0625 * 100    print(f'    - DL improved duplicate recall by {improvement:.0f}%!')if dup_f1 > 0.1031:    f1_improvement = (dup_f1 - 0.1031) / 0.1031 * 100    print(f'    - DL improved duplicate F1-score by {f1_improvement:.0f}%!')

## 21. Download Evaluation Results

In [ ]:
with open('train_embeddings_normalized.npy', 'wb') as f:    np.save(f, train_embeddings_norm)print('Saved train_embeddings_normalized.npy')from google.colab import filesprint('\nDownloading embeddings...')files.download('train_embeddings_normalized.npy')print('Download initiated')

---# SUMMARY---## Training Summary**Model Architecture:**- Embedding(vocab_size, 128, mask_zero=True)- LSTM(64)- Dropout(0.2)- Dense(32, relu)- Dense(4, softmax)**Training Configuration:**- Epochs: 10- Batch size: 32- Validation split: 0.2- Optimizer: Adam- Loss: Categorical crossentropy## Evaluation Summary**Classification:**- Test accuracy: [See Cell 16]- Method: LSTM with softmax**Duplicate Detection:**- Method: Cosine similarity on LSTM embeddings- Threshold: 0.7- Performance: [See Cell 18]## Files to Download**Models:**1. `dl_model.h5` - Trained LSTM model2. `tokenizer.pkl` - Tokenizer with vocabulary3. `label_encoder.pkl` - Label encoder for categories**Embeddings:**4. `train_embeddings_normalized.npy` - Training embeddings for duplicate detection## Next Steps1. Download all 4 files2. Place them in your local `models/` directory3. Use for Streamlit UI or further analysis